# Simple RAG (Retrieval-Augmented Generation) System

## Overview

This code implements a basic Retrieval-Augmented Generation (RAG) system for processing and querying PDF documents.
The system encodes the document content into a vector store, which can then be queried to retrieve relevant information.

## Key Components

1. PDF processing and text extraction
2. Text chunking for manageable processing
3. Vector store creation using FAISS and embeddings
4. Retriever setup for querying the processed documents

## Installation

In [ ]:
!pip install pypdf langchain-core langchain-community langchain-openai langchain-text-splitters faiss-cpu python-dotenv PyMuPDF rank-bm25

## Configuration & Connection Check

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path(os.getcwd()).parent))

from config import get_llm, get_embeddings, LLM_CONFIG, EMBEDDING_CONFIG, check_connections

In [ ]:
# Verify config and test connections
check_connections()

## Imports

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Helper Functions

In [ ]:
def replace_t_with_space(list_of_documents):
    """Replace tab characters with spaces in document content."""
    for doc in list_of_documents:
        doc.page_content = doc.page_content.replace('\t', ' ')
    return list_of_documents


def show_context(context):
    """Display retrieved context chunks."""
    for i, c in enumerate(context):
        print(f"Context {i + 1}:")
        print(c)
        print("\n")

## Load & Encode PDF

In [ ]:
def encode_pdf(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF into a FAISS vector store.

    Args:
        path: Path to the PDF file.
        chunk_size: Size of each text chunk.
        chunk_overlap: Overlap between consecutive chunks.

    Returns:
        A FAISS vector store containing the encoded content.
    """
    # Load PDF
    loader = PyPDFLoader(path)
    documents = loader.load()

    # Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    # Create embeddings and vector store
    embeddings = get_embeddings()
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore

In [ ]:
path = "../data/Understanding_Climate_Change.pdf"
chunks_vector_store = encode_pdf(path, chunk_size=1000, chunk_overlap=200)
print(f"Encoded PDF into vector store with {chunks_vector_store.index.ntotal} vectors.")

## Create Retriever & Test

In [ ]:
chunks_query_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 2})

In [ ]:
test_query = "What is the main cause of climate change?"
docs = chunks_query_retriever.invoke(test_query)
context = [doc.page_content for doc in docs]
show_context(context)

## Answer a Question using RAG

In [ ]:
def answer_question(query, retriever):
    """Retrieve context and generate answer using LLM."""
    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template="""Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context:
{context}

Question: {question}
Answer:"""
    )

    llm = get_llm()
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": query})
    return response.content

In [ ]:
query = "What is the main cause of climate change?"
answer = answer_question(query, chunks_query_retriever)
print(f"Question: {query}")
print(f"Answer: {answer}")

In [ ]:
query2 = "What are the effects of rising sea levels?"
answer2 = answer_question(query2, chunks_query_retriever)
print(f"Question: {query2}")
print(f"Answer: {answer2}")